## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
# importing decision tree classifier 
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

### Reading data

In [2]:
df1 = pd.read_csv ("ML1.csv",header=None)
df1.columns=["reading ID","X_Acc","Y_Acc","Z_Acc","Activity"]

df2 = pd.read_csv ("ML2.csv",header=None)
df2.columns=["reading ID","X_Acc","Y_Acc","Z_Acc","Activity"]

df3 = pd.read_csv ("ML3.csv",header=None)
df3.columns=["reading ID","X_Acc","Y_Acc","Z_Acc","Activity"]

df4 = pd.read_csv ("ML4.csv",header=None)
df4.columns=["reading ID","X_Acc","Y_Acc","Z_Acc","Activity"]

df5 = pd.read_csv ("ML5.csv",header=None)
df5.columns=["reading ID","X_Acc","Y_Acc","Z_Acc","Activity"]



In [3]:
df4.head()

,reading ID,X_Acc,Y_Acc,Z_Acc,Activity
0,0.0,2045,2326,1833,1
1,1.0,2041,2317,1832,1
2,2.0,2046,2323,1836,1
3,3.0,2046,2339,1833,1
4,4.0,2045,2335,1829,1


## Pre-Processing

#### Merging data

In [4]:
df_merge=pd.concat([df1,df2,df3,df4,df5],axis=0)

df_merge.head()

,reading ID,X_Acc,Y_Acc,Z_Acc,Activity
0,0.0,1502,2215,2153,1
1,1.0,1667,2072,2047,1
2,2.0,1611,1957,1906,1
3,3.0,1601,1939,1831,1
4,4.0,1643,1965,1879,1


#### Droping the reading ID

In [5]:
df=df_merge.drop("reading ID",axis=1)
df.head()

,X_Acc,Y_Acc,Z_Acc,Activity
0,1502,2215,2153,1
1,1667,2072,2047,1
2,1611,1957,1906,1
3,1601,1939,1831,1
4,1643,1965,1879,1


In [6]:
print(df.Activity.unique())

[1 2 3 4 5 6 7 0]


In [7]:
df = df[~(df["Activity"]==0)]
df.head()

,X_Acc,Y_Acc,Z_Acc,Activity
0,1502,2215,2153,1
1,1667,2072,2047,1
2,1611,1957,1906,1
3,1601,1939,1831,1
4,1643,1965,1879,1


In [8]:
print(df.Activity.unique())

[1 2 3 4 5 6 7]


In [9]:
data= df.drop_duplicates()
data.reset_index(inplace=True)
data=data.drop('index',axis=1)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 492046 entries, 0 to 492045
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   X_Acc     492046 non-null  int64
 1   Y_Acc     492046 non-null  int64
 2   Z_Acc     492046 non-null  int64
 3   Activity  492046 non-null  int64
dtypes: int64(4)
memory usage: 15.0 MB


In [10]:
y = data['Activity']
data=data.drop('Activity',axis=1)

#### Standardization / Normalization

In [13]:


scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
data_scaled

array([[-4.87457147, -2.34021119,  2.04802244],
       [-3.19525624, -4.62918042,  0.74699539],
       [-3.76520565, -6.46995988, -0.98361605],
       ...,
       [-0.75261591, -0.13127584,  0.63653083],
       [-0.83403725, -0.14728262,  0.78381691],
       [-0.78314891, -0.11526906,  0.57516163]])

In [14]:
data_scaled = pd.DataFrame(data_scaled,
                                 columns=data.columns)
data_scaled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 492046 entries, 0 to 492045
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   X_Acc   492046 non-null  float64
 1   Y_Acc   492046 non-null  float64
 2   Z_Acc   492046 non-null  float64
dtypes: float64(3)
memory usage: 11.3 MB


In [15]:
X=data_scaled

#### Train_Test_Split

In [16]:
#split data into training and testing
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,stratify=y,random_state=10)
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((393636, 3), (393636,), (98410, 3), (98410,))

#### Decision Tree Classifier

In [18]:
model = DecisionTreeClassifier(random_state=10,criterion='gini',class_weight='balanced',max_depth=5)

model.fit(X_train, y_train)
print('Accuracy on training: ',model.score(X_train, y_train))

yhat1 = model.predict(X_test)
print('Accuracy on test data: ',model.score(X_test, y_test))


Accuracy on training:  0.5162993222164639
Accuracy on test data:  0.5165735189513261


In [19]:
from sklearn.model_selection import StratifiedKFold,KFold
kf = StratifiedKFold(n_splits=20,shuffle=True,random_state=10)
acc = []
x=X_train
y=y_train

for fold,(t_id,v_id) in enumerate(kf.split(x,y)):
    tx = x.iloc[t_id]; ty = y.iloc[t_id]
    vx = x.iloc[v_id]; vy = y.iloc[v_id]
    model.fit(tx,ty)
           
    val_y = model.predict(vx)
    acc_score = accuracy_score(vy,val_y)
    acc.append(acc_score)
    print(f"fold {fold} accuracy {acc_score}")

print(f"Mean accuracy score {np.mean(acc_score)}")

fold 0 accuracy 0.5084341022253835
fold 1 accuracy 0.5256071537445381
fold 2 accuracy 0.5185956711716289
fold 3 accuracy 0.5067066355045219
fold 4 accuracy 0.5211868712529214
fold 5 accuracy 0.5371405344985266
fold 6 accuracy 0.5149883141957118
fold 7 accuracy 0.5133116553195813
fold 8 accuracy 0.521745757544965
fold 9 accuracy 0.5041662432679606
fold 10 accuracy 0.5012193882735494
fold 11 accuracy 0.5108728787724824
fold 12 accuracy 0.5135148866985062
fold 13 accuracy 0.49857738034752563
fold 14 accuracy 0.5124987298038817
fold 15 accuracy 0.5157504318666802
fold 16 accuracy 0.5183171586809613
fold 17 accuracy 0.5118134241146283
fold 18 accuracy 0.4991616279660586
fold 19 accuracy 0.5151669122503938
Mean accuracy score 0.5151669122503938


#### Random Forest Classifier

In [20]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(random_state=10,
                            n_estimators=200,class_weight='balanced', n_jobs=-1,
                            )

forest.fit(X_train, y_train)

yhat1 = forest.predict(X_test)

In [21]:
print('Accuracy on training: ',forest.score(X_train, y_train))
print('Accuracy on test data: ',forest.score(X_test, y_test))

Accuracy on training:  0.962724445934823
Accuracy on test data:  0.6815973986383498


In [22]:
kf = StratifiedKFold(n_splits=20,shuffle=True,random_state=10)
acc = []
x=X_train
y=y_train

for fold,(t_id,v_id) in enumerate(kf.split(x,y)):
    tx = x.iloc[t_id]; ty = y.iloc[t_id]
    vx = x.iloc[v_id]; vy = y.iloc[v_id]
    forest.fit(tx,ty)
           
    val_y = forest.predict(vx)
    acc_score = accuracy_score(vy,val_y)
    acc.append(acc_score)
    print(f"fold {fold} accuracy {acc_score}")

print(f"Mean accuracy score {np.mean(acc_score)}")

fold 0 accuracy 0.6815872370694035
fold 1 accuracy 0.6811299664668224
fold 2 accuracy 0.6919520373945737
fold 3 accuracy 0.6781323036276801
fold 4 accuracy 0.6879382176608069
fold 5 accuracy 0.6833147037902653
fold 6 accuracy 0.6809775429326288
fold 7 accuracy 0.6860075195610202
fold 8 accuracy 0.6876841784371507
fold 9 accuracy 0.6869728686109136
fold 10 accuracy 0.6816888527588659
fold 11 accuracy 0.6836703587033838
fold 12 accuracy 0.6801138095721979
fold 13 accuracy 0.684178437150696
fold 14 accuracy 0.685956711716289
fold 15 accuracy 0.6853470175795143
fold 16 accuracy 0.6805040394288908
fold 17 accuracy 0.6813678166759819
fold 18 accuracy 0.6836034754331589
fold 19 accuracy 0.680097556018495
Mean accuracy score 0.680097556018495


In [29]:
df_test = pd.read_csv ("TEST-NewData.csv",header=None)
df_test.columns=["reading ID","X_Acc","Y_Acc","Z_Acc","Activity"]
df_test.head()

,reading ID,X_Acc,Y_Acc,Z_Acc,Activity
0,0.0,1922,2388,1898,1
1,1.0,1880,2327,1832,1
2,2.0,1847,2281,1842,1
3,3.0,1859,2330,1869,1
4,4.0,1931,2306,1853,1


In [30]:
df_test.shape

(114702, 5)

In [31]:
df_test.head(1)

,reading ID,X_Acc,Y_Acc,Z_Acc,Activity
0,0.0,1922,2388,1898,1


In [34]:
df_test.drop(['reading ID','Activity'],axis=1,inplace=True)

In [35]:
y_pred = forest.predict(df_test)

In [43]:
y_pred

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)